# Notebook 3: Experiment 3 — Different Timeframe Training (80/20)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on Weekly/Monthly/Yearly data, predict on Daily test data.  
**Train/Test Split:** 80/20 (chronological, split date from daily data)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Scaler:** ProportionScaler (÷ 10,501)  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

import plotly.graph_objects as go
from plotly.subplots import make_subplots

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'
EXP_LABEL = f'Exp3_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 3 - Different Timeframe Training (80/20)")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 3 - Different Timeframe Training (80/20)


In [3]:
# Load ALL timeframe data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)

print("\nLoading weekly data...")
weekly_data = {}
for stock in STOCKS:
    weekly_data[stock] = load_stock_data(get_data_paths(DATA_DIR, stock, 'weekly'))
    print(f"  {stock}: {len(weekly_data[stock])} records")

print("\nLoading monthly data...")
monthly_data = {}
for stock in STOCKS:
    monthly_data[stock] = load_stock_data(get_data_paths(DATA_DIR, stock, 'monthly'))
    print(f"  {stock}: {len(monthly_data[stock])} records")

print("\nLoading yearly data...")
yearly_data = {}
for stock in STOCKS:
    yearly_data[stock] = load_stock_data(get_data_paths(DATA_DIR, stock, 'yearly'))
    print(f"  {stock}: {len(yearly_data[stock])} records")

print("\nAll data loaded!")

timeframe_data = {
    'weekly': weekly_data,
    'monthly': monthly_data,
    'yearly': yearly_data,
}


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

Loading weekly data...
  TLKM: 1102 records
  BBCA: 1102 records
  ASII: 1102 records
  UNVR: 1102 records

Loading monthly data...
  TLKM: 256 records
  BBCA: 256 records
  ASII: 256 records
  UNVR: 256 records

Loading yearly data...
  TLKM: 22 records
  BBCA: 22 records
  ASII: 22 records
  UNVR: 22 records

All data loaded!


In [4]:
# Reload the module to get the latest fixes
import importlib
import stock_prediction_utils
importlib.reload(stock_prediction_utils)
from stock_prediction_utils import *
print("✓ Module reloaded successfully")

stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
✓ Module reloaded successfully


## Run All Timeframe Experiments

In [5]:
# ============================================================
# EXPERIMENT 3: Different timeframe training
# ============================================================
all_results = []

for stock in STOCKS:
    for tf_name in TIMEFRAMES:
        print(f"\n{'#'*60}")
        print(f"# STOCK: {stock} | TRAIN TIMEFRAME: {tf_name}")
        print(f"{'#'*60}")
        
        train_df = timeframe_data[tf_name][stock]
        test_daily_df = daily_data[stock]
        
        # Prepare data
        X_train, y_train, X_test, y_test, test_dates = prepare_diff_timeframe_data(
            train_df, test_daily_df,
            train_ratio=TRAIN_RATIO, lookback=LOOKBACK
        )
        
        if X_train is None:
            print(f"  SKIPPED: Not enough {tf_name} data for {stock}")
            for model_type in MODEL_TYPES:
                all_results.append({
                    'Stock': stock, 'Train_Timeframe': tf_name,
                    'Model': model_type, 'MSE': np.nan, 'RMSE': np.nan,
                    'MAE': np.nan, 'MAPE (%)': np.nan, 'R2': np.nan,
                    'Note': 'Insufficient training data'
                })
            continue
        
        print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
        
        for model_type in MODEL_TYPES:
            exp_name = f'{EXP_LABEL}_{stock}_{tf_name}'
            
            y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
                model_type=model_type,
                X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                experiment_name=exp_name,
                save_dir=f'models/{EXP_LABEL}',
                epochs=EPOCHS, batch_size=BATCH_SIZE
            )
            
            result = {
                'Stock': stock, 'Train_Timeframe': tf_name,
                'Model': model_type, **metrics
            }
            all_results.append(result)
            
            plot_actual_vs_predicted(
                test_dates, y_true_inv, y_pred_inv,
                model_type, f'{stock}_train_{tf_name}',
                EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
            )

print("\n\nAll Experiment 3 (80/20) training complete!")



############################################################
# STOCK: TLKM | TRAIN TIMEFRAME: weekly
############################################################
  X_train: (877, 1, 1), X_test: (1049, 1, 1)

Training BiLSTM for: Exp3_80_20_TLKM_weekly
  Train samples: 877, Test samples: 1049
Epoch 1/100
11/13 [========================>.....] - ETA: 0s - loss: 0.0169 
Epoch 1: val_loss improved from inf to 0.01825, saving model to models/Exp3_80_20\Exp3_80_20_TLKM_weekly_BiLSTM_best.keras
13/13 [==============================] - 13s 129ms/step - loss: 0.0161 - val_loss: 0.0183
Epoch 2/100
13/13 [==============================] - ETA: 0s - loss: 0.0052
Epoch 2: val_loss improved from 0.01825 to 0.00185, saving model to models/Exp3_80_20\Exp3_80_20_TLKM_weekly_BiLSTM_best.keras
13/13 [==============================] - 0s 21ms/step - loss: 0.0052 - val_loss: 0.0019
Epoch 3/100
11/13 [========================>.....] - ETA: 0s - loss: 0.0042
Epoch 3: val_loss did not improve from 0.00185
13

## Results Summary

In [6]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 3 - Different Timeframe (80/20)")

results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 3 - Different Timeframe (80/20)
Stock Train_Timeframe  Model          MSE      RMSE       MAE  MAPE (%)        R2  Training_Time_s  Epochs_Run
 TLKM          weekly BiLSTM    3131.9160   55.9635   41.7629    1.3499  0.981667             34.5         100
 TLKM          weekly  BiGRU    4432.3362   66.5758   52.2354    1.6617  0.974055             29.0         100
 TLKM          weekly   LSTM    3467.3140   58.8839   44.0896    1.4343  0.979704             22.1         100
 TLKM          weekly    GRU    3011.9681   54.8814   40.7619    1.3248  0.982369             22.3         100
 TLKM         monthly BiLSTM    3593.4176   59.9451   45.8847    1.4779  0.978966             16.5         100
 TLKM         monthly  BiGRU    3040.9617   55.1449   40.9085    1.3267  0.982200             17.2         100
 TLKM         monthly   LSTM    3841.7709   61.9820   47.3800    1.5174  0.977512             10.6         100
 TLKM         monthly    GRU    3376.6376   58.1088   44.0666    1

## Visualizations

In [7]:
# ============================================================
# INTERACTIVE RESULTS VISUALIZATIONS
# ============================================================

print("Generating interactive results visualizations...\n")

# 1. Interactive Metrics Comparison
print("1. Generating Metrics Comparison Chart...")
fig1, html1 = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2']
)
print(f"   ✓ Saved: {html1}")
fig1.show()

print()

# 2. Model Radar Chart
print("2. Generating Model Radar Chart...")
fig2, html2 = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html2}")
fig2.show()

print()

# 3. Interactive Dashboard with Subplots
print("3. Generating Results Dashboard...")
fig3 = go.Figure()

# RMSE by model
rmse_by_model = results_df.groupby('Model')['RMSE'].mean().sort_values()
fig3.add_trace(go.Bar(
    x=rmse_by_model.index,
    y=rmse_by_model.values,
    name='RMSE',
    marker_color='#0072B2',
    text=np.round(rmse_by_model.values, 4),
    textposition='outside',
    hovertemplate='Model: %{x}<br>Avg RMSE: %{y:.4f}<extra></extra>'
))

fig3.update_layout(
    title=f"<b>{EXP_LABEL} - Average Metrics by Model</b>",
    xaxis_title="Model",
    yaxis_title="RMSE",
    height=600,
    template='plotly_white',
    font=dict(size=12),
    showlegend=False
)

html3 = f'figures/{EXP_LABEL}/{EXP_LABEL}_metrics_dashboard.html'
fig3.write_html(html3)
print(f"   ✓ Saved: {html3}")
fig3.show()

print("\n✓ All interactive results visualizations generated successfully!")

Generating interactive results visualizations...

1. Generating Metrics Comparison Chart...
   ✓ Saved: figures/Exp3_80_20/Exp3_80_20_metrics_comparison.html



2. Generating Model Radar Chart...
   ✓ Saved: figures/Exp3_80_20/Exp3_80_20_model_radar.html



3. Generating Results Dashboard...
   ✓ Saved: figures/Exp3_80_20/Exp3_80_20_metrics_dashboard.html



✓ All interactive results visualizations generated successfully!


## Interactive Results Visualizations

In [8]:
# ============================================================
# METRICS BAR CHARTS PER STOCK
# ============================================================
for stock in STOCKS:
    stock_df = results_df[results_df['Stock'] == stock].copy()
    if stock_df.empty:
        continue
    
    for metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:
        fig, ax = plt.subplots(figsize=(12, 6))
        
        timeframes = stock_df['Train_Timeframe'].unique()
        n_tf = len(timeframes)
        n_models = len(MODEL_TYPES)
        bar_width = 0.8 / n_models
        x = np.arange(n_tf)
        
        for i, model_type in enumerate(MODEL_TYPES):
            vals = []
            for tf in timeframes:
                v = stock_df[(stock_df['Model'] == model_type) & 
                             (stock_df['Train_Timeframe'] == tf)][metric]
                vals.append(v.values[0] if len(v) > 0 and not pd.isna(v.values[0]) else 0)
            
            ax.bar(x + i * bar_width, vals, bar_width,
                   label=model_type, color=MODEL_COLORS[model_type],
                   edgecolor='white', linewidth=0.5)
        
        ax.set_xlabel('Training Timeframe', fontsize=13)
        ax.set_ylabel(metric, fontsize=13)
        ax.set_title(f'{EXP_LABEL} | {stock} - {metric} by Timeframe', fontsize=14)
        ax.set_xticks(x + bar_width * (n_models - 1) / 2)
        ax.set_xticklabels(timeframes, fontsize=11)
        ax.legend(fontsize=11)
        fig.tight_layout()
        
        fname = f'figures/{EXP_LABEL}/{stock}_{metric}_by_timeframe.png'.replace('(%)', 'pct')
        save_fig(fig, fname)

print("All bar charts saved!")


  Figure saved: figures/Exp3_80_20/TLKM_RMSE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/TLKM_MAE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/TLKM_MAPE pct_by_timeframe.png
  Figure saved: figures/Exp3_80_20/TLKM_R2_by_timeframe.png
  Figure saved: figures/Exp3_80_20/BBCA_RMSE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/BBCA_MAE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/BBCA_MAPE pct_by_timeframe.png
  Figure saved: figures/Exp3_80_20/BBCA_R2_by_timeframe.png
  Figure saved: figures/Exp3_80_20/ASII_RMSE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/ASII_MAE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/ASII_MAPE pct_by_timeframe.png
  Figure saved: figures/Exp3_80_20/ASII_R2_by_timeframe.png
  Figure saved: figures/Exp3_80_20/UNVR_RMSE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/UNVR_MAE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/UNVR_MAPE pct_by_timeframe.png
  Figure saved: figures/Exp3_80_20/UNVR_R2_by_timeframe.png
All 

In [9]:
# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("  BEST TIMEFRAME PER STOCK (by RMSE)")
print("="*70)
for stock in STOCKS:
    stock_data = results_df[results_df['Stock'] == stock].dropna(subset=['RMSE'])
    if stock_data.empty:
        continue
    best_idx = stock_data['RMSE'].idxmin()
    best = stock_data.loc[best_idx]
    print(f"  {stock}: {best['Train_Timeframe']} + {best['Model']} "
          f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")



  BEST TIMEFRAME PER STOCK (by RMSE)
  TLKM: weekly + GRU (RMSE=54.8814, R²=0.982369)
  BBCA: weekly + LSTM (RMSE=150.5970, R²=0.979017)
  ASII: monthly + BiGRU (RMSE=82.9563, R²=0.981373)
  UNVR: weekly + BiLSTM (RMSE=69.1619, R²=0.994501)
